In [2]:
import pandas as pd
import numpy as np
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Loading TMDB Data Set

In [ ]:
# df = pd.read_csv("../data/TMDB_all_movies.csv")

In [ ]:
# print(df.shape)

In [ ]:
# print(df.columns.tolist())

In [ ]:
# df['vote_count'].describe()

In [ ]:
# import matplotlib.pyplot as plt
# import seaborn as sns

# # Plot histogram with a logarithmic x-axis (adding 1 to handle 0 vote counts)
# plt.figure(figsize=(10, 5))
# sns.histplot(df["vote_count"] + 1, log_scale=True, bins=10, kde=True)
# plt.title("Distribution of Vote Counts (Log Scale)")
# plt.xlabel("Vote Count (Log Scale)")
# plt.ylabel("Number of Movies")
# plt.show()

In [ ]:
# df['vote_count'].quantile([0.90, 0.95, 0.98, 0.99, 0.995, 0.999])

In [ ]:
# df["release_date"] = pd.to_datetime(df["release_date"])

In [ ]:
# Filter for release dates up to the end of 2026
# recent_movies_2026 = (
#     df[(df["release_date"] <= "2026-7-17") & (df["vote_count"] > 7)]
#     .sort_values(by="release_date", ascending=False)
#     .head(20)[["title", "release_date", "vote_average", "vote_count", "imdb_votes", "imdb_rating"]]
#     .reset_index(drop=True)
# )

# Display the filtered table
# recent_movies_2026

In [ ]:
# Count movies matching all three conditions
# count = df[
#     (df["vote_average"] > 5)
#     & (df["imdb_votes"] > 2000)
#     & (df["vote_count"] > 200)
#     & (df["release_date"] <= "2026-08-17")
#     & (df["release_date"] >= "1900-01-01")
# ].shape[0]

# print(f"Total matching movies: {count}")

In [ ]:
# Filter for release dates up to the end of 2026
# recent_movies_2026 = (
#     df[(df["vote_average"] > 5)
#     & (df["imdb_votes"] > 2000)
#     & (df["release_date"] <= "2026-08-17")
#     & (df["release_date"] >= "1900-01-01")]
#     .sort_values(by="release_date", ascending=True)
#     .head(20)[["title", "release_date", "vote_average", "vote_count", "overview"]]
#     .reset_index(drop=True)
# )

# # Display the filtered table
# recent_movies_2026

In [ ]:
# df_filtered = df[
#     ((df["vote_average"] > 6) & (df["vote_count"] > 500) & (df["imdb_votes"] > 5000)) |
#     (df["release_date"] >= "2023-01-01")
# ].copy()

# print(df_filtered.shape)

In [ ]:
# C = df['vote_average'].mean()
# m = 700  # tune this — higher m means you trust vote_average less until a movie has more votes

# def weighted_rating(row, m=m, C=C):
#     v = row['vote_count']
#     R = row['vote_average']
#     return (v / (v + m) * R) + (m / (v + m) * C)

# df['wr_score'] = df.apply(weighted_rating, axis=1)

In [ ]:
# df['wr_score'].describe()

In [ ]:
# bayesian_df = df.sort_values(by="wr_score", ascending=False)

# keyword = "hero"
# matches = bayesian_df[
#     bayesian_df["title"].str.contains(keyword, case=False, na=False)
# ]

# matches[["title", "release_date", "vote_average", "vote_count"]]

In [ ]:
# bayesian_df = df.sort_values(by="wr_score", ascending=False)
# bayesian_filtered_df = (
#     bayesian_df[(bayesian_df["vote_count"] > 500)
#     & (bayesian_df["release_date"] <= "2026-08-17")
#     & (bayesian_df["release_date"] >= "1900-01-01")]
#     .sort_values(by="release_date", ascending=False)
#     .head(15000)[["title", "release_date", "vote_average", "vote_count", "wr_score"]]
#     .reset_index(drop=True)
# )

# # Display the filtered table
# bayesian_filtered_df

In [ ]:
# Search for any title containing a keyword
# keyword = "spider"
# matches = bayesian_filtered_df[
#     bayesian_filtered_df["title"].str.contains(keyword, case=False, na=False)
# ]

# matches[["title", "release_date", "vote_average", "vote_count"]]

In [ ]:
# bayesian_new_filtered_df = (
#     df[(df["wr_score"] >= 4.1) & (df["release_date"] >= "1900-01-01")]
#     .sort_values(by="wr_score", ascending=False)
#     .head(15000)                                   
#     [["title", "release_date", "vote_average", "vote_count", "wr_score"]]
#     .sort_values(by="release_date", ascending=True) 
#     .reset_index(drop=True)
# )

# bayesian_new_filtered_df

# Narrowed Down Data Set + Clean Up

In [ ]:
bayesian_new_filtered_df = (
    df[(df["wr_score"] >= 4.1) & (df["release_date"] >= "1900-01-01")]
    .sort_values(by="wr_score", ascending=False)
    .head(15000)
    [[
        "title",
        "release_date",
        "vote_average",
        "vote_count",
        "wr_score",
        "overview",
        "genres",
        "keywords",
        "cast",
        "director",
        "poster_path",
        "popularity",
        "runtime",
        "tagline",
        "id"          # TMDB movie id: need this for poster fetching + as a stable unique key
    ]]
    .sort_values(by="release_date", ascending=True)
    .reset_index(drop=True)
)

bayesian_new_filtered_df

In [ ]:
bayesian_new_filtered_df[["overview", "genres", "keywords", "cast", "director", "poster_path"]].isnull().sum()

In [ ]:
#Dropping null values
bayesian_new_filtered_df = bayesian_new_filtered_df.dropna(
    subset=["overview", "genres", "cast", "director", "poster_path"]
).reset_index(drop=True)

print(bayesian_new_filtered_df.shape)

In [ ]:
# saved new filtered CSV
bayesian_new_filtered_df.to_csv("tmdb_7000_movies.csv", index=False)

In [4]:
# Read your newly saved 7,500-movie dataset
bayesian_new_filtered_df = pd.read_csv("../data/TMDB_7000_movies.csv")

# Verify the shape
print(bayesian_new_filtered_df.shape)  # Should output (7000, num_columns)

(6987, 15)


In [5]:
bayesian_new_filtered_df["genres_list"] = bayesian_new_filtered_df["genres"].apply(
    lambda x: [g.strip() for g in x.split(",")] if pd.notna(x) else []
)

bayesian_new_filtered_df["cast_list"] = bayesian_new_filtered_df["cast"].apply(
    lambda x: [c.strip() for c in x.split(",")][:3] if pd.notna(x) else []  # top 3 actors
)

bayesian_new_filtered_df["keywords_list"] = bayesian_new_filtered_df["keywords"].apply(
    lambda x: [k.strip() for k in x.split("|")] if pd.notna(x) else []
)

bayesian_new_filtered_df["director_list"] = bayesian_new_filtered_df["director"].apply(
    lambda x: [x.strip()] if pd.notna(x) else []
)

In [6]:
bayesian_new_filtered_df[["title", "genres_list", "cast_list", "keywords_list", "director_list"]].head(3)

,title,genres_list,cast_list,keywords_list,director_list
0,A Trip to the Moon,"[Adventure, Science Fiction, Comedy]","[Georges Méliès, Bleuette Bernon, François Lal...","[moon, based on novel or book, satire, astrono...",[Georges Méliès]
1,The Great Train Robbery,"[Western, Crime, Action, Adventure]","[Gilbert M. Anderson, John Manus Dougherty Sr....","[robbery, robber, dynamite, hold-up robbery, o...",[Edwin S. Porter]
2,The Cabinet of Dr. Caligari,"[Drama, Horror, Thriller, Crime]","[Werner Krauss, Conrad Veidt, Friedrich Fehér]","[insane asylum, black and white, silent film, ...",[Robert Wiene]


# Stemming Movie Overview

In [7]:
# Collapsing multi-word names so it doesn't affect the vectorizing step
def collapse(lst):
    return [item.replace(" ", "") for item in lst]

bayesian_new_filtered_df["genres_collapsed"] = bayesian_new_filtered_df["genres_list"].apply(collapse)
bayesian_new_filtered_df["cast_collapsed"] = bayesian_new_filtered_df["cast_list"].apply(collapse)
bayesian_new_filtered_df["keywords_collapsed"] = bayesian_new_filtered_df["keywords_list"].apply(collapse)
bayesian_new_filtered_df["director_collapsed"] = bayesian_new_filtered_df["director_list"].apply(collapse)

In [8]:
# building the tags column, a one-space-separated string of info
bayesian_new_filtered_df["overview_words"] = bayesian_new_filtered_df["overview"].apply(lambda x: x.split())

def build_tags(row, genre_weight=4, keyword_weight=4, cast_weight=1, director_weight=2):
    tags = (
        row["overview_words"]
        + row["genres_collapsed"] * genre_weight
        + row["keywords_collapsed"] * keyword_weight
        + row["cast_collapsed"] * cast_weight
        + row["director_collapsed"] * director_weight
    )
    return " ".join(tags).lower()

bayesian_new_filtered_df["tags"] = bayesian_new_filtered_df.apply(build_tags, axis=1)

## Stemming + Vectorizing

In [9]:
from nltk.stem.porter import PorterStemmer
ps = PorterStemmer()

def stem(text):
    return " ".join([ps.stem(word) for word in text.split()])

bayesian_new_filtered_df["tags"] = bayesian_new_filtered_df["tags"].apply(stem)

In [10]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cv = CountVectorizer(max_features=5000, stop_words='english')
vectors = cv.fit_transform(bayesian_new_filtered_df["tags"]).toarray()

similarity = cosine_similarity(vectors)
print(similarity.shape)  

(6987, 6987)


### Single Movie Recommend Function

In [11]:
def recommend(movie_title):
    idx = bayesian_new_filtered_df[bayesian_new_filtered_df["title"] == movie_title].index
    if len(idx) == 0:
        print("Movie not found.")
        return
    idx = idx[0]
    distances = similarity[idx]
    movie_indices = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:6] #not 0 since that's itself
    for i in movie_indices:
        print(bayesian_new_filtered_df.iloc[i[0]].title)

recommend("Project Hail Mary")  # or any title you know is in your filtered set

Spaceman
The Cloverfield Paradox
The Astronaut's Wife
2010
Mickey 17


### Problem right now is popularity bias (ex. Harry Potter franchise)

In [12]:
# capping per franchise
import re

movies_df = bayesian_new_filtered_df

def get_franchise_key(title, n_words=2):
    clean = re.sub(r'[^\w\s]', '', title.lower())  # strip punctuation
    words = clean.split()
    if words and words[0] in ("a", "an"):    # drop leading article so it doesn't skew the key
        words = words[1:]
    return " ".join(words[:n_words])

In [13]:
import random

def build_movie_dict(row, wildcard=False):
    d = {
        "id": int(row.id),
        "title": row.title,
        "poster_path": row.poster_path,
        "runtime": float(row.runtime) if pd.notna(row.runtime) else None,
        "vote_average": float(row.vote_average),
        "release_date": str(row.release_date),
        "genres": row.genres_list,
        "overview": row.overview,
        "tagline": row.tagline if pd.notna(row.tagline) else None,
        "cast": row.cast_list,
        "director": row.director_list,
    }
    if wildcard:
        d["wildcard"] = True
    return d

def recommend(movie_title, max_per_franchise=2, n_results=10, include_wildcard=True):
    idx = movies_df[movies_df["title"] == movie_title].index
    if len(idx) == 0:
        return None
    idx = idx[0]
    query_franchise = get_franchise_key(movie_title)

    distances = cosine_similarity(vectors[idx].reshape(1,-1), vectors)[0]

    # Choosing from the top 150
    candidates = sorted(list(enumerate(distances)), reverse=True, key=lambda x: x[1])[1:150]

    results = []
    franchise_counts = {}

    for i, score in candidates:
        row = movies_df.iloc[i]
        franchise = get_franchise_key(row.title)

        if franchise == query_franchise:
            franchise_counts[franchise] = franchise_counts.get(franchise, 0) + 1
            if franchise_counts[franchise] > max_per_franchise:
                continue

        results.append(build_movie_dict(row))
        if len(results) == n_results:
            break

    if include_wildcard and len(results) > 0:
        #Wildcard chosen randomly from 50-150
        wildcard_candidates = candidates[50:150]
        wildcard_pool = [
            i for i, score in wildcard_candidates
            if get_franchise_key(movies_df.iloc[i].title) != query_franchise
        ]

        if wildcard_pool:
            wc_idx = random.choice(wildcard_pool)
            row = movies_df.iloc[wc_idx]
            results[-1] = build_movie_dict(row, wildcard=True)

    return results

In [14]:
movies = recommend("Harry Potter and the Philosopher's Stone") 
for movie in movies:
        print(movie["title"])

Harry Potter and the Goblet of Fire
Harry Potter and the Half-Blood Prince
Oz the Great and Powerful
The NeverEnding Story
Fantastic Beasts: The Crimes of Grindelwald
The School for Good and Evil
Seventh Son
Fantastic Beasts: The Secrets of Dumbledore
Tales from Earthsea
The Craft


# Cold Start Recommendation Function

In [15]:
# Recommending from multiple liked movies
# import numpy as np
# from sklearn.metrics.pairwise import cosine_similarity

def get_coldstart_recommendations(movie_titles, max_per_franchise=2, n_results=10):
    indices = movies_df[movies_df["title"].isin(movie_titles)].index
    if len(indices) == 0:
        return None

    user_vector = vectors[indices].mean(axis=0).reshape(1, -1)
    sim_scores = cosine_similarity(user_vector, vectors)[0]

    candidates = sorted(list(enumerate(sim_scores)), reverse=True, key=lambda x: x[1])
    candidates = [c for c in candidates if c[0] not in indices][:150]

    results = []
    franchise_counts = {}
    seen_franchises_from_input = {get_franchise_key(t) for t in movie_titles}

    for i, score in candidates:
        row = movies_df.iloc[i]
        franchise = get_franchise_key(row.title)

        if franchise in seen_franchises_from_input:
            franchise_counts[franchise] = franchise_counts.get(franchise, 0) + 1
            if franchise_counts[franchise] > max_per_franchise:
                continue

        results.append(build_movie_dict(row))
        if len(results) == n_results:
            break

    return results

In [16]:
movies = get_coldstart_recommendations(["Inception", "The Matrix", "A Silent Voice: The Movie"])
for movie in movies:
        print(movie["title"])

The Matrix Reloaded
Dragon Ball Z: Fusion Reborn
Dragon Ball Z: Broly - Second Coming
Dragon Ball Z: Resurrection 'F'
The Matrix Revolutions
Ghost in the Shell
Stand by Me Doraemon
Dragon Ball Super: Broly
Batman Ninja
Dragon Ball Z: Bojack Unbound


# Watched Before Recommendation Function

In [17]:
def get_director_keys(row):
    # Extracts a set of normalized director names from a row.
    directors = row.get("director_list", [])
    if isinstance(directors, list):
        return {str(d).strip().lower() for d in directors if d}
    elif isinstance(directors, str):
        return {directors.strip().lower()}
    return set()

In [18]:
# Keyword post-boost only for watched-before recommendations
# Convert keyword lists into single space-separated strings
keyword_strings = movies_df["keywords_list"].apply(
    lambda x: " ".join([str(k).lower().replace(" ", "") for k in x])
    if isinstance(x, list)
    else ""
)

# 2. Vectorize keywords (min_df=2 ignores rare 1-off keywords to save memory)
keyword_vectorizer = CountVectorizer(binary=True, min_df=2)
keyword_vectors = keyword_vectorizer.fit_transform(keyword_strings)

In [19]:
def get_watched_before_recommendations(
    movie_titles, max_per_director=3, n_results=30, keyword_weight=0.6):
    
    indices = movies_df[movies_df["title"].isin(movie_titles)].index
    if len(indices) == 0:
        return None

    # 1. Content similarity (overview, cast, keywords)
    user_content_vec = vectors[indices].mean(axis=0).reshape(1, -1)
    content_sim = cosine_similarity(user_content_vec, vectors)[0]

    # 2. Genre similarity
    user_keyword_vec = np.asarray(keyword_vectors[indices].mean(axis=0)).reshape(1, -1)
    keyword_sim = cosine_similarity(user_keyword_vec, keyword_vectors)[0]

    # 3. Combine scores with weight (e.g., 60% general content + 40% genre match)
    combined_scores = (1 - keyword_weight) * content_sim + (keyword_weight) * keyword_sim

    candidates = sorted(
        list(enumerate(combined_scores)), reverse=True, key=lambda x: x[1]
    )
    candidates = [c for c in candidates if c[0] not in indices][:150]

    results = []
    director_counts = {}

    for i, score in candidates:
        row = movies_df.iloc[i]
        directors = get_director_keys(row)

        skip_movie = False
        for director in directors:
            if director_counts.get(director, 0) >= max_per_director:
                skip_movie = True
                break

        if skip_movie:
            continue

        # Increment counts for all directors credited on this movie
        for director in directors:
            director_counts[director] = director_counts.get(director, 0) + 1

        results.append(build_movie_dict(row))
        if len(results) == n_results:
            break

    return results

In [20]:
movies = get_watched_before_recommendations(["The Incredibles", "A Silent Voice: The Movie", "Project Hail Mary"])
for movie in movies:
        print(movie["title"])

My Hero Academia: World Heroes' Mission
My Hero Academia: Heroes Rising
My Hero Academia: Two Heroes
Captain Underpants: The First Epic Movie
Dragon Ball Z: Broly - The Legendary Super Saiyan
Space Pirate Captain Harlock
Incredibles 2
Justice League: Gods and Monsters
Spaceman
The Anthem of the Heart
Dragon Ball Z: The Return of Cooler
Saint Seiya: Legend of Sanctuary
The Fox and the Hound
Ultraman: Rising
Megamind
Stand by Me Doraemon
DC League of Super-Pets
The Iron Giant
Regular Show: The Movie
Dragon Ball Z: Bojack Unbound
One Piece: Stampede
Legend of the Guardians: The Owls of Ga'Hoole
Next Gen
Big Hero 6
Justice League: Doom
Dragon Ball Super: Super Hero
All Star Superman
Meet the Robinsons
Flavors of Youth
The First Slam Dunk


# Downcasting + Joblib

In [ ]:
import joblib
import numpy as np

# Downcast DataFrame
float_cols = movies_df.select_dtypes(include=['float64']).columns
movies_df[float_cols] = movies_df[float_cols].astype('float32')
int_cols = movies_df.select_dtypes(include=['int64']).columns
movies_df[int_cols] = movies_df[int_cols].astype('int32')

# Downcast matrices
vectors = vectors.astype(np.float32)
keyword_vectors = keyword_vectors.astype(np.float32)

# Save files with joblib
joblib.dump(movies_df, '../models/movies_df.joblib', compress=3)
joblib.dump(vectors, '../models/vectors.joblib', compress=3)
joblib.dump(keyword_vectors, '../models/keyword_vectors.joblib', compress=3)